In [1]:
from smolagents import ToolCollection, ToolCallingAgent, OpenAIServerModel
from mcp import StdioServerParameters
import os
import json

In [2]:
os.environ['NEBIUS_API_KEY'] = open('secret.txt', 'r').read().strip()

In [3]:
MODEL_T = "Qwen/Qwen3-235B-A22B-Thinking-2507"
MODEL_I = "Qwen/Qwen3-235B-A22B-Instruct-2507"

model_t = OpenAIServerModel(
    model_id=MODEL_T,
    api_key=os.environ["NEBIUS_API_KEY"],
    api_base="https://api.studio.nebius.com/v1/",
    temperature=0,
)

model_i = OpenAIServerModel(
    model_id=MODEL_I,
    api_key=os.environ["NEBIUS_API_KEY"],
    api_base="https://api.studio.nebius.com/v1/",
    temperature=0,
)

In [4]:
server = StdioServerParameters(
    command="node",
    args=["/mnt/c/Users/Uniholder/Git/BioThings-MCP-Server/build/index.js", "stdio"]
)

In [5]:
with ToolCollection.from_mcp(
        server_parameters=server,
        trust_remote_code=True,
        structured_output=True
    ) as tools:
        # Проверить, что тулы действительно подхватились:
        print([t.name for t in tools.tools])

['get_gene_annotation', 'query_genes', 'get_variant_annotation', 'query_variants', 'batch_gene_query', 'batch_variant_query', 'get_gene_metadata', 'get_variant_metadata', 'get_gene_fields', 'get_variant_fields', 'search_genes_by_pathway', 'search_genes_by_go_term', 'search_variants_by_gene', 'search_pathogenic_variants', 'get_gene_orthologs', 'search_drug_target_genes', 'get_genomic_interval_genes', 'search_variants_by_population_frequency']


In [13]:
def check_tool(tool, **kwargs):
    with ToolCollection.from_mcp(server, trust_remote_code=True, structured_output=False) as tc:
        ex = [t for t in tc.tools if t.name==tool][0]
        out = ex(kwargs if kwargs else {})
        print(out)

In [ ]:
check_tool(, 'EFO_0000305')

In [6]:
SYSTEM_PROMPT = '''
system_prompt_biothings = """
You are a specialized bioinformatics extraction agent designed for the Longevity Sequence-to-Function Knowledge Base.

You are connected **only** to the BioThings MCP Server, which provides access to biomedical knowledge through its native tools (BioThings API family).

You must rely **exclusively** on the tools exposed by this server. Do not fabricate information, and do not use any external APIs.

---

### 🎯 MISSION
Given a human gene or protein name (e.g., NRF2, APOE, SOX2, OCT4), extract **verified biological and functional knowledge** that maps *sequence regions or variants* to *functional outcomes* relevant to **aging, stress resistance, metabolism, regeneration, or lifespan**.

---
Always be concise, structured, and traceable to BioThings data.
"""

'''

In [8]:
gene_symbol = 'APOE'

In [9]:
user_prompt_A = f"""
You are connected exclusively to the BioThings MCP Server. Use only its tools.

GOAL
Resolve the input gene/protein to stable identifiers and fetch a very compact gene-level summary.

INPUT
Gene/protein: {gene_symbol}
Organism: Homo sapiens

TOOLS ALLOWED (examples; use only what's available on this server)
- query_gene / get_gene_by_id (BioThings mygene-like)
- get_xrefs (if available through BioThings)
Disallow: any non-BioThings tools.

RULES
- Return only the primary match for Homo sapiens.
- Keep summary ≤ 600 characters. No abstracts, no full descriptions.
- Include only the most essential IDs (Entrez, Ensembl ENSG, UniProt, HGNC).
- Include official symbol + 5 most relevant aliases max (if present).
- No sequences. No pathways. No variants here.
- If multiple hits, pick the top-ranked human entry; do not return arrays of candidates.

OUTPUT
Return ONLY JSON that follows the schema below. No prose outside JSON.

JSON SCHEMA (A)
{{
  "query": "{gene_symbol}",
  "biothings_A": {{
    "symbol": "<official>",
    "name": "<official full name>",
    "gene_id_entrez": "<string or null>",
    "gene_id_ensembl": "<ENSG or null>",
    "uniprot": "<accession or null>",
    "hgnc_id": "<HGNC:#### or null>",
    "aliases": ["<alias1>", "<alias2>", "..."],
    "summary_compact": "<<=600 chars>"
  }}
}}
"""


In [10]:
def run_query(user_prompt, model, max_steps=10):
    with ToolCollection.from_mcp(
        server_parameters=server,
        trust_remote_code=True,
        structured_output=False
    ) as tools:
        agent = ToolCallingAgent(
            model=model,
            tools=[*tools.tools],
            add_base_tools=False,
            max_steps=max_steps,
        )
        agent.prompt_templates["system_prompt"] = SYSTEM_PROMPT
        return agent.run(user_prompt)

In [11]:
with ToolCollection.from_mcp(
    server_parameters=server,
    trust_remote_code=True,
    structured_output=False
) as tools:
    agent = ToolCallingAgent(
        model=model_i,
        tools=[*tools.tools],
        add_base_tools=False,
        max_steps=10,
    )
    agent.prompt_templates["system_prompt"] = SYSTEM_PROMPT
    result_A = agent.run(user_prompt_A)

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ You are connected exclusively to the BioThings MCP Server. Use only its tools.                                  │
│                                                                                                                 │
│ GOAL                                                                                                            │
│ Resolve the input gene/protein to stable identifiers and fetch a very compact gene-level summary.               │
│                                                                                                                 │
│ INPUT                                                                                                           │
│ Gene/protein: APOE                                                                                              │
│ Organism: Homo sapiens                                                                                          │
│                                                                                                                 │
│ TOOLS ALLOWED (examples; use only what's available on this server)                                              │
│ - query_gene / get_gene_by_id (BioThings mygene-like)                                                           │
│ - get_xrefs (if available through BioThings)                                                                    │
│ Disallow: any non-BioThings tools.                                                                              │
│                                                                                                                 │
│ RULES                                                                                                           │
│ - Return only the primary match for Homo sapiens.                                                               │
│ - Keep summary ≤ 600 characters. No abstracts, no full descriptions.                                            │
│ - Include only the most essential IDs (Entrez, Ensembl ENSG, UniProt, HGNC).                                    │
│ - Include official symbol + 5 most relevant aliases max (if present).                                           │
│ - No sequences. No pathways. No variants here.                                                                  │
│ - If multiple hits, pick the top-ranked human entry; do not return arrays of candidates.                        │
│                                                                                                                 │
│ OUTPUT                                                                                                          │
│ Return ONLY JSON that follows the schema below. No prose outside JSON.                                          │
│                                                                                                                 │
│ JSON SCHEMA (A)                                                                                                 │
│ {                                                                                                               │
│   "query": "APOE",                                                                                              │
│   "biothings_A": {                                                                                              │
│     "symbol": "<official>",                                                                                     │
│     "name": "<official full name>",                                                                             │
│     "gene_id_entrez": "<string or null>",                                                                       │
│     "gene_id_ensembl": "<ENSG or null>",              

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Calling tool: 'query_genes' with arguments: {'q': 'APOE', 'species': 'human', 'size': 1, 'from': 0, 'fields':   │
│ 'symbol,name,entrezgene,ensembl.gene,uniprot,HGNC,alias,summary', 'sort': '', 'facets': ''}                     │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Observations: {
  "took": 17,
  "total": 4,
  "max_score": 150.91031,
  "hits": |
    {
      "HGNC": "613",
      "_id": "348",
      "_score": 150.91031,
      "alias": |
        "AD2",
        "APO-E",
        "ApoE4",
        "LDLCQ5",
        "LPG"
      ],
      "ensembl": {
        "gene": "ENSG00000130203"
      },
      "entrezgene": "348",
      "name": "apolipoprotein E",
      "summary": "The protein encoded by this gene is a major apoprotein of the chylomicron. It binds to a specific
liver and peripheral cell receptor, and is essential for the normal catabolism of triglyceride-rich lipoprotein 
constituents. This gene maps to chromosome 19 in a cluster with the related apolipoprotein C1 and C2 genes. 
Mutations in this gene result in familial dysbetalipoproteinemia, or type III hyperlipoproteinemia (HLP III), in 
which increased plasma cholesterol and triglycerides are the consequence of impaired clearance of chylomicron and 
VLDL remnants. |provided by RefSeq, Jun 2016].",
      "symbol": "APOE",
      "uniprot": {
        "Swiss-Prot": "P02649",
        "TrEMBL": |
          "E9PEV4",
          "A0A0S2Z3D5",
          "H0Y7L5",
          "E7ERP7"
        ]
      }
    }
  ]
}

[Step 1: Duration 4.28 seconds| Input tokens: 4,116 | Output tokens: 75]

...

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 5 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Calling tool: 'final_answer' with arguments: {'answer': '{"query": "APOE", "biothings_A": {"symbol": "APOE",    │
│ "name": "apolipoprotein E", "gene_id_entrez": "348", "gene_id_ensembl": "ENSG00000130203", "uniprot": "P02649", │
│ "hgnc_id": "HGNC:613", "aliases": ["AD2", "APO-E", "ApoE4", "LDLCQ5", "LPG"], "summary_compact": "The protein   │
│ encoded by this gene is a major apoprotein of the chylomicron. It binds to a specific liver and peripheral cell │
│ receptor, and is essential for the normal catabolism of triglyceride-rich lipoprotein constituents. This gene   │
│ maps to chromosome 19 in a cluster with the related apolipoprotein C1 and C2 genes. Mutations in this gene      │
│ result in familial dysbetalipoproteinemia, or type III hyperlipoproteinemia (HLP III), in which increased       │
│ plasma cholesterol and triglycerides are the consequence of impaired clearance of chylomicron and VLDL          │
│ remnants."}}'}                                                                                                  │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Observations: {"query": "APOE", "biothings_A": {"symbol": "APOE", "name": "apolipoprotein E", "gene_id_entrez": 
"348", "gene_id_ensembl": "ENSG00000130203", "uniprot": "P02649", "hgnc_id": "HGNC:613", "aliases": |"AD2", 
"APO-E", "ApoE4", "LDLCQ5", "LPG"], "summary_compact": "The protein encoded by this gene is a major apoprotein of 
the chylomicron. It binds to a specific liver and peripheral cell receptor, and is essential for the normal 
catabolism of triglyceride-rich lipoprotein constituents. This gene maps to chromosome 19 in a cluster with the 
related apolipoprotein C1 and C2 genes. Mutations in this gene result in familial dysbetalipoproteinemia, or type 
III hyperlipoproteinemia (HLP III), in which increased plasma cholesterol and triglycerides are the consequence of 
impaired clearance of chylomicron and VLDL remnants."}}

Final answer: {"query": "APOE", "biothings_A": {"symbol": "APOE", "name": "apolipoprotein E", "gene_id_entrez": 
"348", "gene_id_ensembl": "ENSG00000130203", "uniprot": "P02649", "hgnc_id": "HGNC:613", "aliases": ["AD2", 
"APO-E", "ApoE4", "LDLCQ5", "LPG"], "summary_compact": "The protein encoded by this gene is a major apoprotein of 
the chylomicron. It binds to a specific liver and peripheral cell receptor, and is essential for the normal 
catabolism of triglyceride-rich lipoprotein constituents. This gene maps to chromosome 19 in a cluster with the 
related apolipoprotein C1 and C2 genes. Mutations in this gene result in familial dysbetalipoproteinemia, or type 
III hyperlipoproteinemia (HLP III), in which increased plasma cholesterol and triglycerides are the consequence of 
impaired clearance of chylomicron and VLDL remnants."}}

[Step 5: Duration 13.21 seconds| Input tokens: 23,060 | Output tokens: 1,317]

In [74]:
json.loads(result_A)

{'query': 'NRF2',
 'biothings_A': {'symbol': 'NFE2L2',
  'name': 'NFE2 like bZIP transcription factor 2',
  'gene_id_entrez': '4780',
  'gene_id_ensembl': 'ENSG00000116044',
  'uniprot': 'Q16236',
  'hgnc_id': None,
  'aliases': ['HEBP1', 'IMDDHH', 'NRF2', 'Nrf-2'],
  'summary_compact': 'Transcription factor regulating antioxidant response elements (ARE); controls expression of genes involved in oxidative stress response, inflammation, and free radical detoxification. Central to cellular defense mechanisms and longevity pathways. [RefSeq]'}}

In [12]:
user_prompt_B = f"""
You are connected exclusively to the BioThings MCP Server. Use only the whitelisted tools below.

WHITELISTED_TOOLS
- query_genes
- get_gene_annotation
- get_gene_metadata

BLACKLISTED_TOOLS
- get_gene_fields
- get_variant_fields
- query_variants
- get_variant_annotation
- batch_gene_query
- batch_variant_query
- search_variants_by_gene
- search_pathogenic_variants
- search_variants_by_population_frequency
- get_gene_orthologs
- search_drug_target_genes
- search_genes_by_pathway
- search_genes_by_go_term
- get_genomic_interval_genes
- get_variant_metadata

STOP RULES
- Never call any tool with empty or placeholder arguments.
- Do not introspect schema (no *fields).
- Once the data sufficient to fill the JSON is collected, immediately return final_answer.

GOAL (Step B: Sequence intervals / motifs → function)
For the human input {gene_symbol}, extract a compact list of **protein domains, motifs, and functional regions**
that can be mapped to distinct molecular functions. Choose those that can be related to longevity directly or indirectly, e.g diseases.

DISAMBIGUATION
- If {gene_symbol} is an alias (e.g., "NRF2"), resolve to the official human protein-coding gene.
- Ignore lncRNA or pseudogene hits.

FIELDS TO REQUEST (include only if they exist)
- interpro
- pfam
- prosite
- uniprot.Swiss-Prot
- go (optional context)
- genomic_pos, exons_hg19 (if available; for coordinates)
- homologene or agr.orthologs (optional; for conservation context)

HOW TO USE THESE FIELDS
- From interpro/pfam/prosite/uniprot → extract **domains/motifs**.
- From uniprot or exons_hg19/genomic_pos → add coordinates if they are explicitly present.
- From homologene/agr.orthologs → you may note conservation in a short phrase if directly included in the field
- If a field is absent, ignore it (do not perform extra queries).

CONTENT RULES
- Up to 100 unique regions. Merge duplicates.
- Each region must include:
  • name (“region_or_motif”)
  • data source (“source”: InterPro|Pfam|PROSITE|UniProt)
  • start/end if available, else null
  • concise “function” (≤ 180 chars) describing molecular mechanism:
    e.g., DNA binding, dimerization, Keap1 interaction, transactivation, degradation signal, etc.
- Optional: add note on conservation if field contains that info.

OUTPUT RULES
- Produce only JSON per schema below.
- Omit any commentary or text outside the JSON.
- Keep nulls for missing coordinates or fields.

JSON SCHEMA (B)
{{
  "query": "{gene_symbol}",
  "biothings_B": {{
    "sequence_intervals": [
      {{
        "region_or_motif": "<name>",
        "source": "<InterPro|Pfam|PROSITE|UniProt>",
        "start": <int or null>,
        "end": <int or null>,
        "function": "<<=180 chars>",
        "conservation": "<short optional phrase or null>"
      }}
    ]
  }}
}}
"""

In [94]:
result_B = run_query(user_prompt_B, model_i, 5)

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ You are connected exclusively to the BioThings MCP Server. Use only the whitelisted tools below.                │
│                                                                                                                 │
│ WHITELISTED_TOOLS                                                                                               │
│ - query_genes                                                                                                   │
│ - get_gene_annotation                                                                                           │
│ - get_gene_metadata                                                                                             │
│                                                                                                                 │
│ BLACKLISTED_TOOLS                                                                                               │
│ - get_gene_fields                                                                                               │
│ - get_variant_fields                                                                                            │
│ - query_variants                                                                                                │
│ - get_variant_annotation                                                                                        │
│ - batch_gene_query                                                                                              │
│ - batch_variant_query                                                                                           │
│ - search_variants_by_gene                                                                                       │
│ - search_pathogenic_variants                                                                                    │
│ - search_variants_by_population_frequency                                                                       │
│ - get_gene_orthologs                                                                                            │
│ - search_drug_target_genes                                                                                      │
│ - search_genes_by_pathway                                                                                       │
│ - search_genes_by_go_term                                                                                       │
│ - get_genomic_interval_genes                                                                                    │
│ - get_variant_metadata                                                                                          │
│                                                                                                                 │
│ STOP RULES                                                                                                      │
│ - Never call any tool with empty or placeholder arguments.                                                      │
│ - Do not introspect schema (no *fields).                                                                        │
│ - Once the data sufficient to fill the JSON is collected, immediately return final_answer.                      │
│                                                                                                                 │
│ GOAL (Step B: Sequence intervals / motifs → function)                                                           │
│ For the human input NRF2, extract a compact list of **protein domains, motifs, and functional regions**         │
│ that can be mapped to distinct molecular functions. Choose those that can be related to longevity directly or   │
│ indirectly, e.g diseases.                             

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Calling tool: 'query_genes' with arguments: {'q': 'NRF2', 'species': 'human', 'size': 1, 'from': 0, 'fields':   │
│ 'symbol,name,entrezgene', 'sort': '', 'facets': ''}                                                             │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Observations: {
  "took": 25,
  "total": 5,
  "max_score": 103.51513,
  "hits": |
    {
      "_id": "128071539",
      "_score": 103.51513,
      "entrezgene": "128071539",
      "name": "NRF2 regulation associated lncRNA",
      "symbol": "NRAL"
    }
  ]
}

[Step 1: Duration 5.49 seconds| Input tokens: 4,486 | Output tokens: 60]

...

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 5 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Calling tool: 'final_answer' with arguments: {'answer': '{"query": "NRF2", "biothings_B":                       │
│ {"sequence_intervals": [{"region_or_motif": "Basic-leucine zipper domain", "source": "InterPro", "start": null, │
│ "end": null, "function": "DNA binding and dimerization for transcriptional activation of antioxidant response   │
│ elements.", "conservation": "Conserved in vertebrates including mouse, rat, zebrafish."}, {"region_or_motif":   │
│ "bZIP_Maf", "source": "InterPro", "start": null, "end": null, "function": "Specific DNA binding and             │
│ heterodimerization with small Maf proteins.", "conservation": "Conserved in vertebrates including mouse, rat,   │
│ zebrafish."}, {"region_or_motif": "Transcription factor, Skn-1-like, DNA-binding domain", "source": "InterPro", │
│ "start": null, "end": null, "function": "DNA binding to antioxidant response elements (ARE) in target gene      │
│ promoters.", "conservation": "Conserved in vertebrates including mouse, rat, zebrafish."}, {"region_or_motif":  │
│ "NFE2-like", "source": "InterPro", "start": null, "end": null, "function": "Core regulatory domain for          │
│ oxidative stress response and longevity-associated gene regulation.", "conservation": "Conserved in vertebrates │
│ including mouse, rat, zebrafish."}, {"region_or_motif": "Basic-leucine zipper domain superfamily", "source":    │
│ "InterPro", "start": null, "end": null, "function": "Mediates dimerization and DNA binding as part of the bZIP  │
│ transcription factor family.", "conservation": "Conserved in vertebrates including mouse, rat,                  │
│ zebrafish."}]}}'}                                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Observations: {"query": "NRF2", "biothings_B": {"sequence_intervals": |{"region_or_motif": "Basic-leucine zipper 
domain", "source": "InterPro", "start": null, "end": null, "function": "DNA binding and dimerization for 
transcriptional activation of antioxidant response elements.", "conservation": "Conserved in vertebrates including 
mouse, rat, zebrafish."}, {"region_or_motif": "bZIP_Maf", "source": "InterPro", "start": null, "end": null, 
"function": "Specific DNA binding and heterodimerization with small Maf proteins.", "conservation": "Conserved in 
vertebrates including mouse, rat, zebrafish."}, {"region_or_motif": "Transcription factor, Skn-1-like, DNA-binding 
domain", "source": "InterPro", "start": null, "end": null, "function": "DNA binding to antioxidant response 
elements (ARE) in target gene promoters.", "conservation": "Conserved in vertebrates including mouse, rat, 
zebrafish."}, {"region_or_motif": "NFE2-like", "source": "InterPro", "start": null, "end": null, "function": "Core 
regulatory domain for oxidative stress response and longevity-associated gene regulation.", "conservation": 
"Conserved in vertebrates including mouse, rat, zebrafish."}, {"region_or_motif": "Basic-leucine zipper domain 
superfamily", "source": "InterPro", "start": null, "end": null, "function": "Mediates dimerization and DNA binding 
as part of the bZIP transcription factor family.", "conservation": "Conserved in vertebrates including mouse, rat, 
zebrafish."}]}}

Final answer: {"query": "NRF2", "biothings_B": {"sequence_intervals": [{"region_or_motif": "Basic-leucine zipper 
domain", "source": "InterPro", "start": null, "end": null, "function": "DNA binding and dimerization for 
transcriptional activation of antioxidant response elements.", "conservation": "Conserved in vertebrates including 
mouse, rat, zebrafish."}, {"region_or_motif": "bZIP_Maf", "source": "InterPro", "start": null, "end": null, 
"function": "Specific DNA binding and heterodimerization with small Maf proteins.", "conservation": "Conserved in 
vertebrates including mouse, rat, zebrafish."}, {"region_or_motif": "Transcription factor, Skn-1-like, DNA-binding 
domain", "source": "InterPro", "start": null, "end": null, "function": "DNA binding to antioxidant response 
elements (ARE) in target gene promoters.", "conservation": "Conserved in vertebrates including mouse, rat, 
zebrafish."}, {"region_or_motif": "NFE2-like", "source": "InterPro", "start": null, "end": null, "function": "Core 
regulatory domain for oxidative stress response and longevity-associated gene regulation.", "conservation": 
"Conserved in vertebrates including mouse, rat, zebrafish."}, {"region_or_motif": "Basic-leucine zipper domain 
superfamily", "source": "InterPro", "start": null, "end": null, "function": "Mediates dimerization and DNA binding 
as part of the bZIP transcription factor family.", "conservation": "Conserved in vertebrates including mouse, rat, 
zebrafish."}]}}

[Step 5: Duration 16.66 seconds| Input tokens: 46,405 | Output tokens: 1,944]

In [96]:
json.loads(result_B)

{'query': 'NRF2',
 'biothings_B': {'sequence_intervals': [{'region_or_motif': 'Basic-leucine zipper domain',
    'source': 'InterPro',
    'start': None,
    'end': None,
    'function': 'DNA binding and dimerization for transcriptional activation of antioxidant response elements.',
    'conservation': 'Conserved in vertebrates including mouse, rat, zebrafish.'},
   {'region_or_motif': 'bZIP_Maf',
    'source': 'InterPro',
    'start': None,
    'end': None,
    'function': 'Specific DNA binding and heterodimerization with small Maf proteins.',
    'conservation': 'Conserved in vertebrates including mouse, rat, zebrafish.'},
   {'region_or_motif': 'Transcription factor, Skn-1-like, DNA-binding domain',
    'source': 'InterPro',
    'start': None,
    'end': None,
    'function': 'DNA binding to antioxidant response elements (ARE) in target gene promoters.',
    'conservation': 'Conserved in vertebrates including mouse, rat, zebrafish.'},
   {'region_or_motif': 'NFE2-like',
    'source'

In [ ]:
official_symbol = "NFE2L2"
entrez_id = "4780"

user_prompt_C_phase1 = f"""
You are connected exclusively to the BioThings MCP Server.

GOAL
Discover ClinVar-backed variant candidates for the human gene {official_symbol} (Entrez:{entrez_id}) with minimal fields and no downstream processing.

ALLOWED_TOOLS
- query_variants (exactly one call)

QUERY
Call `query_variants` with:
  q: "clinvar.gene.symbol:{official_symbol} AND _exists_:clinvar.clinical_significance"
  fields: "_id,rsid,hgvs,clinvar.gene.symbol,clinvar.clinical_significance,clinvar.conditions.name,clinvar.review_status,vep.most_severe_consequence,cadd.phred,gnomad.af"
  size: 120
  from: 0
  sort: "-cadd.phred"

POST-PROCESS (in-memory only)
- Keep only entries where clinvar.gene.symbol == "{official_symbol}".
- Drop duplicates by rsid (fallback key: hgvs if rsid is null).
- Do NOT re-rank; preserve API order.

OUTPUT (JSON only)
{{
  "query": "{official_symbol}",
  "biothings_C_phase1": {{
    "meta": {{
      "entrez_id": "{entrez_id}",
      "query_used": "clinvar.gene.symbol:{official_symbol} AND _exists_:clinvar.clinical_significance"
    }},
    "candidates": [
      {{
        "variant_id": "<rs#### or _id if rsid null>",
        "hgvs": "<string or null>",
        "clinvar_significance": "<string or null>",
        "condition": "<single trait or null>",
        "review_status": "<string or null>",
        "consequence": "<vep.most_severe_consequence or null>",
        "cadd_phred": <float or null>,
        "af": <float or null>
      }}
    ]
  }}
}}
"""


In [69]:
official_symbol = "NFE2L2"
entrez_id = "4780"

user_prompt_C_phase1 = f"""
You are connected exclusively to the BioThings MCP Server.

────────────────────────
SCOPE (Phase 1 only)
────────────────────────
Goal: discover a compact, de-duplicated candidate set of variants for {official_symbol} (Entrez:{entrez_id}) with minimal fields,
without doing any deep annotation or synthesis. DO NOT call batch_variant_query or get_variant_annotation in this phase.

────────────────────────
WHITELISTED_TOOLS
────────────────────────
- query_variants
- search_variants_by_gene  # fallback only

BLACKLISTED_TOOLS
- get_variant_annotation
- batch_variant_query
- query_genes
- get_variant_fields
- get_gene_fields
- get_gene_annotation
- get_gene_metadata
- batch_gene_query
- get_variant_metadata
- search_genes_by_pathway
- search_genes_by_go_term
- get_gene_orthologs
- get_genomic_interval_genes
- search_drug_target_genes
- search_pathogenic_variants
- search_variants_by_population_frequency

────────────────────────
STOP RULES
────────────────────────
- Never use fields='all'.
- Never call tools with empty/placeholder args.
- Always use {official_symbol} / Entrez:{entrez_id} as the target gene.
- Perform at most ONE tool call per substep below; if a call errors (HTTP 400), immediately apply the specified retry.
- As soon as you can fill the JSON at the end, return final_answer. No extra commentary outside JSON.

────────────────────────
PHASE 1 — Candidate Discovery (with safe fallbacks)
────────────────────────
Run the following substeps sequentially until you obtain a NON-EMPTY result set (hits > 0). Keep size small.

A1) query_variants
    q: dbnsfp.genename:{official_symbol}
    size: 80, from: 0
    fields:
      _id, rsid, hgvs,
      dbnsfp.genename, snpeff.ann.gene_name,
      vep.most_severe_consequence, snpeff.ann.effect,
      cadd.phred,
      gnomad.af, gnomad_exome.af, gnomad_genome.af,
      clinvar.clinical_significance
    sort: -cadd.phred
    If 400 → retry same call WITHOUT 'sort'.
    If still 400 → retry with reduced fields:
      _id, rsid, hgvs, dbnsfp.genename, snpeff.ann.gene_name, vep.most_severe_consequence, cadd.phred, gnomad.af, clinvar.clinical_significance

B) If A* empty → query_variants
    q: snpeff.ann.gene_name:{official_symbol}
    Use the same fields/sort and the same 400-retry policy as in A1.

C) If still empty → query_variants
    q: clinvar.gene.symbol:{official_symbol}
    Use the same fields/sort and the same 400-retry policy as in A1.

D) If still empty → fallback search_variants_by_gene
    {{"gene_symbol":"{official_symbol}","size":60}}
    (Do not set 'clinical_significance' or 'variant_types' here.)

E) If any step returns total >> 2000, immediately rerun that same step with ONE extra filter to shrink:
    Either add: _exists_:clinvar.clinical_significance
    OR add: cadd.phred:[10 TO *]
    Keep size=80.

────────────────────────
LOCAL POST-FILTER (in memory; do not send as q)
────────────────────────
1) Mapping to {official_symbol}:
   Keep a variant if ANY of:
   - dbnsfp.genename == "{official_symbol}"  OR
   - snpeff.ann.gene_name contains "{official_symbol}"  OR
   - clinvar.gene.symbol == "{official_symbol}" (if present in the record).

2) Consequence extraction:
   - most_severe_consequence := vep.most_severe_consequence if present,
     else derive the heaviest from snpeff.ann.effect using priority:
       stop_gained > frameshift_variant > splice_acceptor_variant > splice_donor_variant > start_lost > missense_variant
   - Keep only variants whose final consequence is one of:
     {{missense_variant, stop_gained, splice_acceptor_variant, splice_donor_variant, frameshift_variant, start_lost}}

3) Frequency aggregation:
   - af := first non-null of [gnomad.af, gnomad_exome.af, gnomad_genome.af], else null.

4) De-duplication:
   - Prefer unique rsid where available; otherwise unique by (chrom:pos:ref:alt) or unique hgvs.
   - If multiple alts at same position produce the same consequence, keep the one with higher cadd.phred, or the one having clinvar.clinical_significance.

5) Ranking (for this phase only; no synthesis):
   - Primary: presence of clinvar.clinical_significance
   - Secondary: consequence severity (use the priority above)
   - Tertiary: higher cadd.phred
   - Quaternary: lower af

6) Limit:
   - Take top ≤ 60 candidates AFTER filtering/deduplication (this phase may output more than 20; diversity quotas are applied in Phase 2).

────────────────────────
OUTPUT
────────────────────────
Return ONLY JSON per the schema below. Use nulls for missing fields. No extra text.

JSON SCHEMA (Phase 1)
{{
  "query": "{official_symbol}",
  "biothings_C_phase1": {{
    "meta": {{
      "entrez_id": "{entrez_id}",
      "query_used": "<the q of the first successful substep (A/B/C/D) or the fallback label>",
      "total_hits_initial": <int>,
      "total_after_filters": <int>
    }},
    "candidates": [
      {{
        "variant_id": "<rs#### if present else _id or chr:g…>",
        "rsid": "<string or null>",
        "hgvs": "<string or null>",
        "maps_to_gene": <true|false>,
        "map_source": "<dbnsfp.genename|snpeff.ann.gene_name|clinvar.gene.symbol>",
        "most_severe_consequence": "<string or null>",
        "raw_effects": ["<zero or more snpeff.ann.effect terms (unique)>"],
        "cadd_phred": <float or null>,
        "af": <float or null>,
        "has_clinvar": <true|false>
      }}
    ]
  }}
}}
"""


In [70]:
result_C1a = run_query(user_prompt_C_phase1, 10)

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ You are connected exclusively to the BioThings MCP Server.                                                      │
│                                                                                                                 │
│ ────────────────────────                                                                                        │
│ SCOPE (Phase 1 only)                                                                                            │
│ ────────────────────────                                                                                        │
│ Goal: discover a compact, de-duplicated candidate set of variants for NFE2L2 (Entrez:4780) with minimal fields, │
│ without doing any deep annotation or synthesis. DO NOT call batch_variant_query or get_variant_annotation in    │
│ this phase.                                                                                                     │
│                                                                                                                 │
│ ────────────────────────                                                                                        │
│ WHITELISTED_TOOLS                                                                                               │
│ ────────────────────────                                                                                        │
│ - query_variants                                                                                                │
│ - search_variants_by_gene  # fallback only                                                                      │
│                                                                                                                 │
│ BLACKLISTED_TOOLS                                                                                               │
│ - get_variant_annotation                                                                                        │
│ - batch_variant_query                                                                                           │
│ - query_genes                                                                                                   │
│ - get_variant_fields                                                                                            │
│ - get_gene_fields                                                                                               │
│ - get_gene_annotation                                                                                           │
│ - get_gene_metadata                                                                                             │
│ - batch_gene_query                                                                                              │
│ - get_variant_metadata                                                                                          │
│ - search_genes_by_pathway                                                                                       │
│ - search_genes_by_go_term                                                                                       │
│ - get_gene_orthologs                                                                                            │
│ - get_genomic_interval_genes                                                                                    │
│ - search_drug_target_genes                                                                                      │
│ - search_pathogenic_variants                                                                                    │
│ - search_variants_by_population_frequency                                                                       │
│                                                       

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Calling tool: 'query_variants' with arguments: {'q': 'dbnsfp.genename:NFE2L2', 'size': 80, 'from': 0, 'fields': │
│ '_id,rsid,hgvs,dbnsfp.genename,snpeff.ann.gene_name,vep.most_severe_consequence,snpeff.ann.effect,cadd.phred,gn │
│ omad.af,gnomad_exome.af,gnomad_genome.af,clinvar.clinical_significance', 'sort': '-cadd.phred', 'facets': ''}   │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Observations: {
  "took": 266,
  "total": 4751,
  "max_score": null,
  "hits": |
    {
      "_id": "chr2:g.178097139A>T",
      "_score": null,
      "cadd": {
        "_license": "http://bit.ly/2TIuab9",
        "phred": 45
      },
      "dbnsfp": {
        "_license": "http://bit.ly/2VLnQBz",
        "genename": |
          "NFE2L2",
          "NFE2L2",
          "NFE2L2",
          "NFE2L2",
          "NFE2L2",
          "NFE2L2",
          "NFE2L2",
          "NFE2L2"
        ]
      },
      "snpeff": {
        "_license": "http://bit.ly/2suyRKt",
        "ann": |
          {
            "effect": "stop_gained"
          },
          {
            "effect": "stop_gained"
          },
          {
            "effect": "stop_gained"
          },
          {
            "effect": "stop_gained"
          },
          {
            "effect": "stop_gained"
          },
          {
            "effect": "stop_gained"
          },
          {
            "effect": "stop_gained"
          },
          {
            "effect": "stop_gained"
          }
        ]
      }
    },
    {
      "_id": "chr2:g.178097139A>C",
      "_score": null,
      "cadd": {
        "_license": "http://bit.ly/2TIuab9",
        "phred": 44
      },
      "dbnsfp": {
        "_license": "http://bit.ly/2VLnQBz",
        "genename": |
          "NFE2L2",
          "NFE2L2",
          "NFE2L2",
          "NFE2L2",
          "NFE2L2",
          "NFE2L2",
          "NFE2L2",
          "NFE2L2"
        ]
      },
      "snpeff": {
        "_license": "http://bit.ly/2suyRKt",
        "ann": |
          {
            "effect": "stop_gained"
          },
          {
            "effect": "stop_gained"
          },
          {
            "effect": "stop_gained"
          },
          {
            "effect": "stop_gained"
          },
          {
            "effect": "stop_gained"
          },
          {
            "effect": "stop_gained"
          },
          {
            "effect": "stop_gained"
          },
          {
            "effect": "stop_gained"
          }
        ]
      }
    },
    {
      "_id": "chr2:g.178095652A>T",
      "_score": null,
      "cadd": {
        "_license": "http://bit.ly/2TIuab9",
        "phred": 44
      },
      "dbnsfp": {
        "_license": "http://bit.ly/2VLnQBz",
        "genename": |
          "NFE2L2",
          "NFE2L2",
          "NFE2L2",
          "NFE2L2"
        ]
      },
      "snpeff": {
        "_license": "http://bit.ly/2suyRKt",
        "ann": |
          {
            "effect": "stop_gained"
          },
          {
            "effect": "stop_gained"
          },
          {
            "effect": "stop_gained"
          },
          {
            "effect": "stop_gained"
          },
          {
            "effect": "stop_gained"
          },
          {
            "effect": "stop_gained"
          },
          {
            "effect": "stop_gained"
          },
          {
            "effect": "stop_gained"
          }
        ]
      }
    },
    {
      "_id": "chr2:g.178095652A>C",
      "_score": null,
      "cadd": {
        "_license": "http://bit.ly/2TIuab9",
        "phred": 43
      },
      "dbnsfp": {
        "_license": "http://bit.ly/2VLnQBz",
        "genename": |
          "NFE2L2",
          "NFE2L2",
          "NFE2L2",
          "NFE2L2"
        ]
      },
      "snpeff": {
        "_license": "http://bit.ly/2suyRKt",
        "ann": |
          {
            "effect": "stop_gained"
          },
          {
            "effect": "stop_gained"
          },
          {
            "effect": "stop_gained"
          },
          {
            "effect": "stop_gained"
          },
          {
            "effect": "stop_gained"
          },
          {
            "effect": "stop_gained"
          },
          {
            "effect": "stop_gained"
          },
          {
            "effect": "stop_gained"
          }
        ]
      }
    },
    {
      "_id": "chr2:g.178095745A>T",
 

[Step 1: Duration 5.85 seconds| Input tokens: 5,230 | Output tokens: 131]

...

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 3 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Calling tool: 'final_answer' with arguments: {'answer': '{"query": "NFE2L2", "biothings_C_phase1": {"meta":     │
│ {"entrez_id": "4780", "query_used": "dbnsfp.genename:NFE2L2", "total_hits_initial": 4751,                       │
│ "total_after_filters": 80}, "candidates": [{"variant_id": "chr2:g.178097139A>T", "rsid": null, "hgvs": null,    │
│ "maps_to_gene": true, "map_source": "dbnsfp.genename", "most_severe_consequence": "stop_gained", "raw_effects": │
│ ["stop_gained"], "cadd_phred": 45, "af": null, "has_clinvar": false}, {"variant_id": "chr2:g.178097139A>C",     │
│ "rsid": null, "hgvs": null, "maps_to_gene": true, "map_source": "dbnsfp.genename", "most_severe_consequence":   │
│ "stop_gained", "raw_effects": ["stop_gained"], "cadd_phred": 44, "af": null, "has_clinvar": false},             │
│ {"variant_id": "chr2:g.178095652A>T", "rsid": null, "hgvs": null, "maps_to_gene": true, "map_source":           │
│ "dbnsfp.genename", "most_severe_consequence": "stop_gained", "raw_effects": ["stop_gained"], "cadd_phred": 44,  │
│ "af": null, "has_clinvar": false}, {"variant_id": "chr2:g.178095652A>C", "rsid": null, "hgvs": null,            │
│ "maps_to_gene": true, "map_source": "dbnsfp.genename", "most_severe_consequence": "stop_gained", "raw_effects": │
│ ["stop_gained"], "cadd_phred": 43, "af": null, "has_clinvar": false}, {"variant_id": "chr2:g.178095745A>T",     │
│ "rsid": null, "hgvs": null, "maps_to_gene": true, "map_source": "dbnsfp.genename", "most_severe_consequence":   │
│ "stop_gained", "raw_effects": ["stop_gained"], "cadd_phred": 43, "af": null, "has_clinvar": false},             │
│ {"variant_id": "chr2:g.178095715A>T", "rsid": null, "hgvs": null, "maps_to_gene": true, "map_source":           │
│ "dbnsfp.genename", "most_severe_consequence": "stop_gained", "raw_effects": ["stop_gained"], "cadd_phred": 42,  │
│ "af": null, "has_clinvar": false}, {"variant_id": "chr2:g.178095736A>T", "rsid": null, "hgvs": null,            │
│ "maps_to_gene": true, "map_source": "dbnsfp.genename", "most_severe_consequence": "stop_gained", "raw_effects": │
│ ["stop_gained"], "cadd_phred": 42, "af": null, "has_clinvar": false}, {"variant_id": "chr2:g.178095614C>A",     │
│ "rsid": null, "hgvs": null, "maps_to_gene": true, "map_source": "dbnsfp.genename", "most_severe_consequence":   │
│ "stop_gained", "raw_effects": ["stop_gained"], "cadd_phred": 42, "af": null, "has_clinvar": false},             │
│ {"variant_id": "chr2:g.178095773C>A", "rsid": null, "hgvs": null, "maps_to_gene": true, "map_source":           │
│ "dbnsfp.genename", "most_severe_consequence": "stop_gained", "raw_effects": ["stop_gained"], "cadd_phred": 42,  │
│ "af": null, "has_clinvar": false}, {"variant_id": "chr2:g.178095841A>T", "rsid": null, "hgvs": null,            │
│ "maps_to_gene": true, "map_source": "dbnsfp.genename", "most_severe_consequence": "stop_gained", "raw_effects": │
│ ["stop_gained", "sequence_feature"], "cadd_phred": 42, "af": null, "has_clinvar": false}, {"variant_id":        │
│ "chr2:g.178098039A>T", "rsid": null, "hgvs": null, "maps_to_gene": true, "map_source": "dbnsfp.genename",       │
│ "most_severe_consequence": "stop_gained", "raw_effects": ["stop_gained", "intron_variant"], "cadd_phred": 42,   │
│ "af": null, "has_clinvar": false}, {"variant_id": "chr2:g.178095884C>A", "rsid": null, "hgvs": null,            │
│ "maps_to_gene": true, "map_source": "dbnsfp.genename", "most_severe_consequence": "stop_gained", "raw_effects": │
│ ["stop_gained"], "cadd_phred": 42, "af": null, "has_clinvar": false}, {"variant_id": "chr2:g.178095644C>A",     │
│ "rsid": null, "hgvs": null, "maps_to_gene": true, "map_source": "dbnsfp.genename", "most_severe_consequence":   │
│ "stop_gained", "raw_effects": ["stop_gained"], "cadd_phred": 42, "af": null, "has_clinvar": false},             │
│ {"variant_id": "chr2:g.178095728C>A", "rsid": null, "h

Observations: {"query": "NFE2L2", "biothings_C_phase1": {"meta": {"entrez_id": "4780", "query_used": 
"dbnsfp.genename:NFE2L2", "total_hits_initial": 4751, "total_after_filters": 80}, "candidates": |{"variant_id": 
"chr2:g.178097139A>T", "rsid": null, "hgvs": null, "maps_to_gene": true, "map_source": "dbnsfp.genename", 
"most_severe_consequence": "stop_gained", "raw_effects": |"stop_gained"], "cadd_phred": 45, "af": null, 
"has_clinvar": false}, {"variant_id": "chr2:g.178097139A>C", "rsid": null, "hgvs": null, "maps_to_gene": true, 
"map_source": "dbnsfp.genename", "most_severe_consequence": "stop_gained", "raw_effects": |"stop_gained"], 
"cadd_phred": 44, "af": null, "has_clinvar": false}, {"variant_id": "chr2:g.178095652A>T", "rsid": null, "hgvs": 
null, "maps_to_gene": true, "map_source": "dbnsfp.genename", "most_severe_consequence": "stop_gained", 
"raw_effects": |"stop_gained"], "cadd_phred": 44, "af": null, "has_clinvar": false}, {"variant_id": 
"chr2:g.178095652A>C", "rsid": null, "hgvs": null, "maps_to_gene": true, "map_source": "dbnsfp.genename", 
"most_severe_consequence": "stop_gained", "raw_effects": |"stop_gained"], "cadd_phred": 43, "af": null, 
"has_clinvar": false}, {"variant_id": "chr2:g.178095745A>T", "rsid": null, "hgvs": null, "maps_to_gene": true, 
"map_source": "dbnsfp.genename", "most_severe_consequence": "stop_gained", "raw_effects": |"stop_gained"], 
"cadd_phred": 43, "af": null, "has_clinvar": false}, {"variant_id": "chr2:g.178095715A>T", "rsid": null, "hgvs": 
null, "maps_to_gene": true, "map_source": "dbnsfp.genename", "most_severe_consequence": "stop_gained", 
"raw_effects": |"stop_gained"], "cadd_phred": 42, "af": null, "has_clinvar": false}, {"variant_id": 
"chr2:g.178095736A>T", "rsid": null, "hgvs": null, "maps_to_gene": true, "map_source": "dbnsfp.genename", 
"most_severe_consequence": "stop_gained", "raw_effects": |"stop_gained"], "cadd_phred": 42, "af": null, 
"has_clinvar": false}, {"variant_id": "chr2:g.178095614C>A", "rsid": null, "hgvs": null, "maps_to_gene": true, 
"map_source": "dbnsfp.genename", "most_severe_consequence": "stop_gained", "raw_effects": |"stop_gained"], 
"cadd_phred": 42, "af": null, "has_clinvar": false}, {"variant_id": "chr2:g.178095773C>A", "rsid": null, "hgvs": 
null, "maps_to_gene": true, "map_source": "dbnsfp.genename", "most_severe_consequence": "stop_gained", 
"raw_effects": |"stop_gained"], "cadd_phred": 42, "af": null, "has_clinvar": false}, {"variant_id": 
"chr2:g.178095841A>T", "rsid": null, "hgvs": null, "maps_to_gene": true, "map_source": "dbnsfp.genename", 
"most_severe_consequence": "stop_gained", "raw_effects": |"stop_gained", "sequence_feature"], "cadd_phred": 42, 
"af": null, "has_clinvar": false}, {"variant_id": "chr2:g.178098039A>T", "rsid": null, "hgvs": null, 
"maps_to_gene": true, "map_source": "dbnsfp.genename", "most_severe_consequence": "stop_gained", "raw_effects": 
|"stop_gained", "intron_variant"], "cadd_phred": 42, "af": null, "has_clinvar": false}, {"variant_id": 
"chr2:g.178095884C>A", "rsid": null, "hgvs": null, "maps_to_gene": true, "map_source": "dbnsfp.genename", 
"most_severe_consequence": "stop_gained", "raw_effects": |"stop_gained"], "cadd_phred": 42, "af": null, 
"has_clinvar": false}, {"variant_id": "chr2:g.178095644C>A", "rsid": null, "hgvs": null, "maps_to_gene": true, 
"map_source": "dbnsfp.genename", "most_severe_consequence": "stop_gained", "raw_effects": |"stop_gained"], 
"cadd_phred": 42, "af": null, "has_clinvar": false}, {"variant_id": "chr2:g.178095728C>A", "rsid": null, "hgvs": 
null, "maps_to_gene": true, "map_source": "dbnsfp.genename", "most_severe_consequence": "stop_gained", 
"raw_effects": |"stop_gained"], "cadd_phred": 41, "af": null, "has_clinvar": false}, {"variant_id": 
"chr2:g.178095980C>A", "rsid": null, "hgvs": null, "maps_to_gene": true, "map_source": "dbnsfp.genename", 
"most_severe_consequence": "stop_gained", "raw_effects": |"stop_gained"], "cadd_phred": 41, "af": null, 
"has_clinvar": false}, {"varia

Final answer: {"query": "NFE2L2", "biothings_C_phase1": {"meta": {"entrez_id": "4780", "query_used": 
"dbnsfp.genename:NFE2L2", "total_hits_initial": 4751, "total_after_filters": 80}, "candidates": [{"variant_id": 
"chr2:g.178097139A>T", "rsid": null, "hgvs": null, "maps_to_gene": true, "map_source": "dbnsfp.genename", 
"most_severe_consequence": "stop_gained", "raw_effects": ["stop_gained"], "cadd_phred": 45, "af": null, 
"has_clinvar": false}, {"variant_id": "chr2:g.178097139A>C", "rsid": null, "hgvs": null, "maps_to_gene": true, 
"map_source": "dbnsfp.genename", "most_severe_consequence": "stop_gained", "raw_effects": ["stop_gained"], 
"cadd_phred": 44, "af": null, "has_clinvar": false}, {"variant_id": "chr2:g.178095652A>T", "rsid": null, "hgvs": 
null, "maps_to_gene": true, "map_source": "dbnsfp.genename", "most_severe_consequence": "stop_gained", 
"raw_effects": ["stop_gained"], "cadd_phred": 44, "af": null, "has_clinvar": false}, {"variant_id": 
"chr2:g.178095652A>C", "rsid": null, "hgvs": null, "maps_to_gene": true, "map_source": "dbnsfp.genename", 
"most_severe_consequence": "stop_gained", "raw_effects": ["stop_gained"], "cadd_phred": 43, "af": null, 
"has_clinvar": false}, {"variant_id": "chr2:g.178095745A>T", "rsid": null, "hgvs": null, "maps_to_gene": true, 
"map_source": "dbnsfp.genename", "most_severe_consequence": "stop_gained", "raw_effects": ["stop_gained"], 
"cadd_phred": 43, "af": null, "has_clinvar": false}, {"variant_id": "chr2:g.178095715A>T", "rsid": null, "hgvs": 
null, "maps_to_gene": true, "map_source": "dbnsfp.genename", "most_severe_consequence": "stop_gained", 
"raw_effects": ["stop_gained"], "cadd_phred": 42, "af": null, "has_clinvar": false}, {"variant_id": 
"chr2:g.178095736A>T", "rsid": null, "hgvs": null, "maps_to_gene": true, "map_source": "dbnsfp.genename", 
"most_severe_consequence": "stop_gained", "raw_effects": ["stop_gained"], "cadd_phred": 42, "af": null, 
"has_clinvar": false}, {"variant_id": "chr2:g.178095614C>A", "rsid": null, "hgvs": null, "maps_to_gene": true, 
"map_source": "dbnsfp.genename", "most_severe_consequence": "stop_gained", "raw_effects": ["stop_gained"], 
"cadd_phred": 42, "af": null, "has_clinvar": false}, {"variant_id": "chr2:g.178095773C>A", "rsid": null, "hgvs": 
null, "maps_to_gene": true, "map_source": "dbnsfp.genename", "most_severe_consequence": "stop_gained", 
"raw_effects": ["stop_gained"], "cadd_phred": 42, "af": null, "has_clinvar": false}, {"variant_id": 
"chr2:g.178095841A>T", "rsid": null, "hgvs": null, "maps_to_gene": true, "map_source": "dbnsfp.genename", 
"most_severe_consequence": "stop_gained", "raw_effects": ["stop_gained", "sequence_feature"], "cadd_phred": 42, 
"af": null, "has_clinvar": false}, {"variant_id": "chr2:g.178098039A>T", "rsid": null, "hgvs": null, 
"maps_to_gene": true, "map_source": "dbnsfp.genename", "most_severe_consequence": "stop_gained", "raw_effects": 
["stop_gained", "intron_variant"], "cadd_phred": 42, "af": null, "has_clinvar": false}, {"variant_id": 
"chr2:g.178095884C>A", "rsid": null, "hgvs": null, "maps_to_gene": true, "map_source": "dbnsfp.genename", 
"most_severe_consequence": "stop_gained", "raw_effects": ["stop_gained"], "cadd_phred": 42, "af": null, 
"has_clinvar": false}, {"variant_id": "chr2:g.178095644C>A", "rsid": null, "hgvs": null, "maps_to_gene": true, 
"map_source": "dbnsfp.genename", "most_severe_consequence": "stop_gained", "raw_effects": ["stop_gained"], 
"cadd_phred": 42, "af": null, "has_clinvar": false}, {"variant_id": "chr2:g.178095728C>A", "rsid": null, "hgvs": 
null, "maps_to_gene": true, "map_source": "dbnsfp.genename", "most_severe_consequence": "stop_gained", 
"raw_effects": ["stop_gained"], "cadd_phred": 41, "af": null, "has_clinvar": false}, {"variant_id": 
"chr2:g.178095980C>A", "rsid": null, "hgvs": null, "maps_to_gene": true, "map_source": "dbnsfp.genename", 
"most_severe_consequence": "stop_gained", "raw_effects": ["stop_gained"], "cadd_phred": 41, "af": null, 
"has_clinvar": false}, {"varia

[Step 3: Duration 178.21 seconds| Input tokens: 86,990 | Output tokens: 8,404]

In [73]:
json.loads(result_C)

{'query': 'NFE2L2',
 'biothings_C_phase1': {'meta': {'entrez_id': '4780',
   'query_used': 'dbnsfp.genename:NFE2L2',
   'total_hits_initial': 4751,
   'total_after_filters': 80},
  'candidates': [{'variant_id': 'chr2:g.178097139A>T',
    'rsid': None,
    'hgvs': None,
    'maps_to_gene': True,
    'map_source': 'dbnsfp.genename',
    'most_severe_consequence': 'stop_gained',
    'raw_effects': ['stop_gained'],
    'cadd_phred': 45,
    'af': None,
    'has_clinvar': False},
   {'variant_id': 'chr2:g.178097139A>C',
    'rsid': None,
    'hgvs': None,
    'maps_to_gene': True,
    'map_source': 'dbnsfp.genename',
    'most_severe_consequence': 'stop_gained',
    'raw_effects': ['stop_gained'],
    'cadd_phred': 44,
    'af': None,
    'has_clinvar': False},
   {'variant_id': 'chr2:g.178095652A>T',
    'rsid': None,
    'hgvs': None,
    'maps_to_gene': True,
    'map_source': 'dbnsfp.genename',
    'most_severe_consequence': 'stop_gained',
    'raw_effects': ['stop_gained'],
    'cadd_p

In [76]:
official_symbol = "NFE2L2"
entrez_id = "4780"

user_prompt_C_phase1b = f"""
You are connected exclusively to the BioThings MCP Server.

────────────────────────
SCOPE (Phase 1b — enrichment only)
────────────────────────
Goal: augment the existing Phase 1 candidate set for {official_symbol} (Entrez:{entrez_id}) with:
  (a) ClinVar-annotated variants, and
  (b) non-LoF consequences (missense, splice, frameshift, start_lost).
Use ONLY the tools listed below. DO NOT call deep annotation tools (no batch_variant_query / get_variant_annotation).

You have access to the Phase 1 JSON in the chat context (field 'biothings_C_phase1.candidates'). 
Treat those as EXISTING_CANDIDATES and DO NOT return duplicates by rsid or hgvs or chr:g key.

────────────────────────
WHITELISTED_TOOLS
────────────────────────
- query_variants

BLACKLISTED_TOOLS
- get_variant_annotation
- batch_variant_query
- search_variants_by_gene
- query_genes
- get_gene_fields
- get_gene_annotation
- get_gene_metadata
- get_variant_fields
- get_variant_metadata
- batch_gene_query
- search_genes_by_pathway
- search_genes_by_go_term
- get_gene_orthologs
- get_genomic_interval_genes
- search_drug_target_genes
- search_pathogenic_variants
- search_variants_by_population_frequency

────────────────────────
STOP & ERROR RULES
────────────────────────
- Never use fields='all'.
- Never call tools with empty/placeholder args.
- Always target {official_symbol} / Entrez:{entrez_id}.
- At most ONE tool call per substep; if HTTP 400, immediately retry as specified.
- As soon as you can fill the JSON at the end, return final_answer (no extra commentary).

────────────────────────
PHASE 1b — Enrichment Queries (run sequentially; collect union; deduplicate vs EXISTING_CANDIDATES)
────────────────────────
Use size small-to-moderate to avoid overfetch. Prefer rsID/hgvs fields.

Q1) ClinVar-enriched set
  query_variants:
    q: (dbnsfp.genename:{official_symbol} OR snpeff.ann.gene_name:{official_symbol} OR clinvar.gene.symbol:{official_symbol}) AND _exists_:clinvar.clinical_significance
    size: 60, from: 0
    fields:
      _id, rsid, hgvs,
      dbnsfp.genename, snpeff.ann.gene_name,
      vep.most_severe_consequence, snpeff.ann.effect,
      clinvar.clinical_significance, clinvar.conditions.name, clinvar.review_status, clinvar.accession,
      cadd.phred,
      gnomad.af
    sort: -cadd.phred
  If 400 → retry without sort.
  If still 400 → reduce fields to:
      _id, rsid, hgvs, dbnsfp.genename, snpeff.ann.gene_name, vep.most_severe_consequence, clinvar.clinical_significance, cadd.phred, gnomad.af

Q2) Missense-focused set
  query_variants:
    q: (dbnsfp.genename:{official_symbol} OR snpeff.ann.gene_name:{official_symbol})
       AND (vep.most_severe_consequence:missense_variant OR snpeff.ann.effect:missense_variant)
    size: 80, from: 0
    fields:
      _id, rsid, hgvs,
      dbnsfp.genename, snpeff.ann.gene_name,
      vep.most_severe_consequence, snpeff.ann.effect,
      cadd.phred,
      gnomad.af,
      clinvar.clinical_significance
    sort: -cadd.phred
  400 → retry without sort → if 400 → drop snpeff.ann.effect.

Q3) Splice/frameshift/start_lost set
  query_variants:
    q: (dbnsfp.genename:{official_symbol} OR snpeff.ann.gene_name:{official_symbol})
       AND (vep.most_severe_consequence:(splice_acceptor_variant OR splice_donor_variant OR frameshift_variant OR start_lost)
            OR snpeff.ann.effect:(splice_acceptor_variant OR splice_donor_variant OR frameshift_variant OR start_lost))
    size: 80, from: 0
    fields:
      _id, rsid, hgvs,
      dbnsfp.genename, snpeff.ann.gene_name,
      vep.most_severe_consequence, snpeff.ann.effect,
      cadd.phred,
      gnomad.af,
      clinvar.clinical_significance
    sort: -cadd.phred
  400 → retry without sort → if 400 → drop snpeff.ann.effect.

If any step returns total > 2000 → rerun same step with extra filter cadd.phred:[10 TO *] and size=80.

────────────────────────
LOCAL POST-PROCESS (do NOT send these as q)
────────────────────────
Mapping to {official_symbol}: keep if any of:
  - dbnsfp.genename == "{official_symbol}" OR
  - snpeff.ann.gene_name contains "{official_symbol}" OR
  - clinvar.gene.symbol == "{official_symbol}" (if present).

Consequence:
  most_severe_consequence := vep.most_severe_consequence if present,
  else heaviest from snpeff.ann.effect (priority: stop_gained > frameshift_variant > splice_acceptor_variant > splice_donor_variant > start_lost > missense_variant).

Keep only consequences in {{missense_variant, stop_gained, splice_acceptor_variant, splice_donor_variant, frameshift_variant, start_lost}}.

AF:
  af := first non-null of [gnomad.af] (else null). Do NOT coerce unknown to 0.0.

De-duplication (two stages):
  1) Remove duplicates *within* Phase 1b across Q1/Q2/Q3 by rsid, then by hgvs, else by chr:g key (_id if that is chr:g).
  2) Remove anything that already exists in Phase 1 (EXISTING_CANDIDATES) by the same keys.

Ranking (for reporting only):
  presence of clinvar.clinical_significance > consequence severity > higher cadd.phred > lower af.

Limit:
  Return up to 120 newly found candidates (after deduplication vs Phase 1).

────────────────────────
OUTPUT (ONLY JSON; no extra text)
────────────────────────
JSON SCHEMA (Phase 1b)
{{
  "query": "{official_symbol}",
  "biothings_C_phase1b": {{
    "meta": {{
      "entrez_id": "{entrez_id}",
      "augmented_from_phase1": true,
      "q1_total": <int>,
      "q2_total": <int>,
      "q3_total": <int>,
      "new_candidates_after_dedup": <int>
    }},
    "new_candidates": [
      {{
        "variant_id": "<rs#### if present else _id or chr:g…>",
        "rsid": "<string or null>",
        "hgvs": "<string or null>",
        "source_bucket": "<Q1_ClinVar|Q2_Missense|Q3_Splice_Frameshift>",
        "maps_to_gene": <true|false>,
        "map_source": "<dbnsfp.genename|snpeff.ann.gene_name|clinvar.gene.symbol>",
        "most_severe_consequence": "<string or null>",
        "raw_effects": ["<0+ snpeff.ann.effect terms>"],
        "cadd_phred": <float or null>,
        "af": <float or null>,
        "has_clinvar": <true|false>
      }}
    ]
  }}
}}
"""


In [80]:
result_C1b = run_query(user_prompt_C_phase1b, 4)

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ You are connected exclusively to the BioThings MCP Server.                                                      │
│                                                                                                                 │
│ ────────────────────────                                                                                        │
│ SCOPE (Phase 1b — enrichment only)                                                                              │
│ ────────────────────────                                                                                        │
│ Goal: augment the existing Phase 1 candidate set for NFE2L2 (Entrez:4780) with:                                 │
│   (a) ClinVar-annotated variants, and                                                                           │
│   (b) non-LoF consequences (missense, splice, frameshift, start_lost).                                          │
│ Use ONLY the tools listed below. DO NOT call deep annotation tools (no batch_variant_query /                    │
│ get_variant_annotation).                                                                                        │
│                                                                                                                 │
│ You have access to the Phase 1 JSON in the chat context (field 'biothings_C_phase1.candidates').                │
│ Treat those as EXISTING_CANDIDATES and DO NOT return duplicates by rsid or hgvs or chr:g key.                   │
│                                                                                                                 │
│ ────────────────────────                                                                                        │
│ WHITELISTED_TOOLS                                                                                               │
│ ────────────────────────                                                                                        │
│ - query_variants                                                                                                │
│                                                                                                                 │
│ BLACKLISTED_TOOLS                                                                                               │
│ - get_variant_annotation                                                                                        │
│ - batch_variant_query                                                                                           │
│ - search_variants_by_gene                                                                                       │
│ - query_genes                                                                                                   │
│ - get_gene_fields                                                                                               │
│ - get_gene_annotation                                                                                           │
│ - get_gene_metadata                                                                                             │
│ - get_variant_fields                                                                                            │
│ - get_variant_metadata                                                                                          │
│ - batch_gene_query                                                                                              │
│ - search_genes_by_pathway                                                                                       │
│ - search_genes_by_go_term                                                                                       │
│ - get_gene_orthologs                                  

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Calling tool: 'query_variants' with arguments: {'q': '(dbnsfp.genename:NFE2L2 OR snpeff.ann.gene_name:NFE2L2 OR │
│ clinvar.gene.symbol:NFE2L2) AND _exists_:clinvar.clinical_significance', 'size': 60, 'from': 0, 'fields':       │
│ '_id,rsid,hgvs,dbnsfp.genename,snpeff.ann.gene_name,vep.most_severe_consequence,snpeff.ann.effect,clinvar.clini │
│ cal_significance,clinvar.conditions.name,clinvar.review_status,clinvar.accession,cadd.phred,gnomad.af', 'sort': │
│ '-cadd.phred', 'facets': ''}                                                                                    │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Observations: {
  "took": 442,
  "total": 0,
  "max_score": null,
  "hits": |]
}

[Step 1: Duration 12.27 seconds| Input tokens: 5,527 | Output tokens: 169]

...

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Calling tool: 'query_variants' with arguments: {'q': '(dbnsfp.genename:NFE2L2 OR snpeff.ann.gene_name:NFE2L2)   │
│ AND (vep.most_severe_consequence:(splice_acceptor_variant OR splice_donor_variant OR frameshift_variant OR      │
│ start_lost) OR snpeff.ann.effect:(splice_acceptor_variant OR splice_donor_variant OR frameshift_variant OR      │
│ start_lost))', 'size': 80, 'from': 0, 'fields':                                                                 │
│ '_id,rsid,hgvs,dbnsfp.genename,snpeff.ann.gene_name,vep.most_severe_consequence,snpeff.ann.effect,cadd.phred,gn │
│ omad.af,clinvar.clinical_significance', 'sort': '-cadd.phred', 'facets': ''}                                    │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Observations: {
  "took": 75,
  "total": 20,
  "max_score": null,
  "hits": |
    {
      "_id": "chr2:g.178098813C>A",
      "_score": null,
      "cadd": {
        "_license": "http://bit.ly/2TIuab9",
        "phred": 38
      },
      "dbnsfp": {
        "_license": "http://bit.ly/2VLnQBz",
        "genename": |
          "NFE2L2",
          "NFE2L2",
          "NFE2L2",
          "NFE2L2",
          "NFE2L2",
          "NFE2L2",
          "NFE2L2",
          "NFE2L2",
          "NFE2L2",
          "NFE2L2"
        ]
      },
      "snpeff": {
        "_license": "http://bit.ly/2suyRKt",
        "ann": |
          {
            "effect": "stop_gained"
          },
          {
            "effect": "stop_gained"
          },
          {
            "effect": "stop_gained"
          },
          {
            "effect": "stop_gained"
          },
          {
            "effect": "stop_gained"
          },
          {
            "effect": "stop_gained"
          },
          {
            "effect": "start_lost"
          },
          {
            "effect": "protein_protein_contact"
          },
          {
            "effect": "protein_protein_contact"
          },
          {
            "effect": "sequence_feature"
          },
          {
            "effect": "sequence_feature"
          },
          {
            "effect": "intron_variant"
          }
        ]
      }
    },
    {
      "_id": "chr2:g.178098813C>T",
      "_score": null,
      "cadd": {
        "_license": "http://bit.ly/2TIuab9",
        "phred": 34
      },
      "dbnsfp": {
        "_license": "http://bit.ly/2VLnQBz",
        "genename": |
          "NFE2L2",
          "NFE2L2",
          "NFE2L2",
          "NFE2L2",
          "NFE2L2",
          "NFE2L2",
          "NFE2L2",
          "NFE2L2",
          "NFE2L2",
          "NFE2L2"
        ]
      },
      "snpeff": {
        "_license": "http://bit.ly/2suyRKt",
        "ann": |
          {
            "effect": "start_lost"
          },
          {
            "effect": "protein_protein_contact"
          },
          {
            "effect": "protein_protein_contact"
          },
          {
            "effect": "missense_variant"
          },
          {
            "effect": "missense_variant"
          },
          {
            "effect": "missense_variant"
          },
          {
            "effect": "missense_variant"
          },
          {
            "effect": "missense_variant"
          },
          {
            "effect": "missense_variant"
          },
          {
            "effect": "sequence_feature"
          },
          {
            "effect": "sequence_feature"
          },
          {
            "effect": "intron_variant"
          }
        ]
      }
    },
    {
      "_id": "chr2:g.178098815T>C",
      "_score": null,
      "cadd": {
        "_license": "http://bit.ly/2TIuab9",
        "phred": 29
      },
      "dbnsfp": {
        "_license": "http://bit.ly/2VLnQBz",
        "genename": |
          "NFE2L2",
          "NFE2L2",
          "NFE2L2",
          "NFE2L2",
          "NFE2L2",
          "NFE2L2",
          "NFE2L2",
          "NFE2L2",
          "NFE2L2",
          "NFE2L2"
        ]
      },
      "snpeff": {
        "_license": "http://bit.ly/2suyRKt",
        "ann": |
          {
            "effect": "start_lost"
          },
          {
            "effect": "missense_variant"
          },
          {
            "effect": "missense_variant"
          },
          {
            "effect": "missense_variant"
          },
          {
            "effect": "missense_variant"
          },
          {
            "effect": "missense_variant"
          },
          {
            "effect": "missense_variant"
          },
          {
            "effect": "intron_variant"
          }
        ]
      }
    },
    {
      "_id": "chr2:g.178098995A>T",
      "_score": null,
      "cadd": {
        "_license": "http://bit.ly/2TIuab9",
        "phred": 28.3
      

[Step 4: Duration 22.46 seconds| Input tokens: 48,283 | Output tokens: 680]

Reached max steps.

[Step 5: Duration 405.58 seconds| Input tokens: 83,592 | Output tokens: 8,872]

In [ ]:
json.loads(result_C1b)